<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/02_searching_ped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/02_searching_ped.ipynb).

# 2. Searching PED

PED holds over 25,000 entries: a few hundred experimentally derived ensembles, plus large numbers of molecular dynamics ensembles and structure-based predictions (AlphaFlex, AF-CALVADOS, IDPForge and others). Finding the right ones is usually the first job.

`ped.search()` returns identifiers and `ped.search_entries()` returns full `PEDEntry` records. Both take the same criteria, which PED combines with AND, and both page through the results automatically. Nothing is downloaded here: searches only return metadata.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

from collections import Counter

## Free text

The first positional argument is PED's free-text search across the whole entry: titles, protein names, methods, cross-references and so on.

In [ ]:
print(ped.search("sic1"))

Free text can be broad. `max_results` stops the search early, which matters for common words.

In [ ]:
print(ped.search("SAXS", max_results=10))

## By protein

`protein_name` and `uniprot` target the constructs in the entry. UniProt accessions are the most reliable way to collect every entry for a protein.

In [ ]:
print("tau:            ", ped.search(protein_name="tau")[:12], "...")
print("alpha-synuclein:", ped.search(uniprot="P37840"))

## By method

`term` matches the IDPO ontology terms PED attaches to each entry: the experimental methods (NMR, SAXS, smFRET, ...) and ensemble-generation methods.

In [ ]:
for method in ["NMR", "SAXS", "smFRET", "PRE"]:
    ids = ped.search(term=method)
    print(f"{method:7s} {len(ids):5d} entries, e.g. {ids[:4]}")

## By publication

`publication` takes a PubMed ID or DOI.

In [ ]:
print(ped.search(publication="20399186"))

## Combining criteria

Criteria combine with AND. For example, alpha-synuclein entries that used SAXS:

In [ ]:
print(ped.search(uniprot="P37840", term="SAXS"))

## Ensemble identifiers

With `ensembles=True`, `search()` returns one identifier per *ensemble*, ready to pass to `ped.load_ensemble()`.

In [ ]:
print(ped.search("sic1", ensembles=True, max_results=4))

## Summarising results with `search_entries()`

`search_entries()` returns `PEDEntry` records, so you can filter and summarise without downloading anything. Here is every alpha-synuclein entry, with its ensembles, their sizes, and the methods used.

In [ ]:
asyn = ped.search_entries(uniprot="P37840")

print(f"{'entry':10s} {'ensembles (conformers)':26s} {'title'}")
for e in asyn:
    ensembles = ", ".join(f"{s.ensemble_id} ({s.n_models})" for s in e.ensembles)
    print(f"{e.entry_id:10s} {ensembles:26s} {e.title[:60]}")

Entries from experiments carry ontology terms for their methods, while many of the large-scale predictions carry none. That makes it easy to split them, or to count which methods have been used for a protein.

In [ ]:
experimental = [e for e in asyn if e.ontology_terms]
predicted = [e for e in asyn if not e.ontology_terms]
print(f"{len(experimental)} entries with method annotations, {len(predicted)} without")

counts = Counter(term.strip() for e in experimental for term in e.ontology_terms)
top = counts.most_common(12)

fig, ax = plt.subplots(figsize=(3.4, 3.2))
ax.barh([t for t, n in top][::-1], [n for t, n in top][::-1], color="black")
ax.set_xlabel("Alpha-synuclein entries (count)")
plt.tight_layout()
plt.show()

## Two quirks of PED's search

* `author` matches PED's *data owner* (the depositor), not the paper's author list, so searching for an author usually finds nothing. Use free text instead: `ped.search("Forman-Kay")`.
* `cross_ref` currently returns a server error from PED (HTTP 500). A free-text search for the identifier works: `ped.search("DP00631")`.

Also note that a search with **no** criteria returns every entry in PED: over 25,000 records, which takes several minutes. Always narrow it down, or use `max_results`.

In [ ]:
print("free text 'Forman-Kay':", ped.search("Forman-Kay", max_results=8))
print("free text 'DP00631':   ", ped.search("DP00631"))